In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import polars as pl
import torch
from torch.utils.data import DataLoader, TensorDataset

from fart.model.device import get_device
from fart.model.nbeats import NBeatsNet
from fart.model.nbeats_config import NBeatsConfig
from fart.model.nbeats_dataset import build_return_windows
from fart.model.train_model import prepare_training_data
from fart.utils import get_project_root
from fart.visualization.confidence_calibration import plot_confidence_calibration
from fart.visualization.plot_styles import apply_plot_styles

apply_plot_styles()

In [ ]:
assets_dir = get_project_root() / "assets"
market, interval = "BTC-EUR", "1d"

config = NBeatsConfig()
lookback = config.lookback

X_train, X_test, y_train, y_test = prepare_training_data(
    data_dir=assets_dir,
    market=market,
    interval=interval,
    months=None,
)

n_train = y_train.shape[0]
close_prices = pl.concat([y_train, y_test])
X_all, y_all = build_return_windows(close_prices, lookback)

n_train_windows = max(0, n_train - lookback - 1)
X_train_windows, y_train_windows = X_all[:n_train_windows], y_all[:n_train_windows]
X_test_windows, y_test_windows = X_all[n_train_windows:], y_all[n_train_windows:]

len(close_prices), X_train_windows.shape, X_test_windows.shape

In [ ]:
device = get_device()
num_members = 5

member_predictions: list[torch.Tensor] = []

for seed in range(num_members):
    torch.manual_seed(seed)
    model = NBeatsNet(config).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=config.learning_rate)

    train_loader = DataLoader(
        TensorDataset(X_train_windows, y_train_windows),
        batch_size=config.batch_size,
        shuffle=True,
    )

    model.train()
    epoch_loss = 0.0
    for epoch in range(config.epochs):
        epoch_loss = 0.0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            mu, _log_sigma = model(X_batch).unbind(-1)
            loss = torch.nn.functional.mse_loss(mu, y_batch)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * X_batch.shape[0]

    model.eval()
    with torch.no_grad():
        mu_test, _log_sigma_test = model(X_test_windows.to(device)).unbind(-1)
    member_predictions.append(mu_test.cpu())

    print(f"member {seed}: final epoch loss = {epoch_loss / len(X_train_windows):.6f}")

predictions = torch.stack(member_predictions)
predictions.shape

In [ ]:
mu_ensemble = predictions.mean(dim=0).numpy()
sigma_ensemble = predictions.std(dim=0).numpy()

error = np.abs(y_test_windows.numpy() - mu_ensemble)
confidence = 1 / (1 + sigma_ensemble)

print(f"confidence range: [{confidence.min():.5f}, {confidence.max():.5f}]")
print(f"error range: [{error.min():.6f}, {error.max():.6f}]")

In [ ]:
plot_confidence_calibration(confidence, error)

## Comparison to notebook 2.0

| | Self-reported confidence (notebook 2.0, single run) | Ensemble disagreement (this notebook, single run) |
|---|---|---|
| n (test windows) | 535 | 535 |
| Pearson r | +0.043 | -0.202 |
| Confidence range | [0.965, 0.976] (0.011 wide) | [0.94495, 0.99776] (0.053 wide) |

**Caveat: this is a single ensemble run.** Unlike notebook 3.0's 30-run reproducibility study of the self-reported approach, there is no run-to-run variance estimate for ensemble disagreement yet, so the numbers above are one data point, not a settled result.

**Leading finding: the confidence spread widened substantially.** Notebook 2.0's self-reported confidence collapsed to a narrow 0.011-wide band ([0.965, 0.976]); this run's ensemble disagreement spans a much wider 0.053-wide range ([0.94495, 0.99776]). This is the more trustworthy result of the two, because it's a structural property of the estimator — an ensemble of independently-trained models genuinely disagrees more on harder windows — rather than a statistic computed on one run and sensitive to that run's noise.

**Correlation: correctly signed, magnitude not yet established.** Notebook 2.0's r=+0.043 is not the right baseline to compare against — notebook 3.0's 30-run study found the self-reported approach's *mean* Pearson r is actually ≈ -0.09, meaning +0.043 was itself a single favorable draw, not a stable figure. Two things follow from comparing against the right baseline:

- **Sign**: notebook 2.0's single-run r was positive — the theoretically wrong direction, since higher confidence should predict *lower* error. This run's ensemble r (-0.202) is negative — the theoretically correct direction. Notebook 3.0's 30-run mean (-0.09) was also negative, so notebook 2.0's positive draw was itself the outlier; still, within this notebook's single run, the ensemble's sign is unambiguously the correct one and notebook 2.0's reported sign was not.
- **Magnitude**: -0.202 is larger in magnitude than both the +0.043 single draw and the ≈-0.09 30-run mean, but with only one ensemble run there's no way yet to tell whether that's a genuine effect or a favorable draw from ensemble disagreement's own (unmeasured) noise distribution — the same mistake this project already made once and retracted (the −0.044→−0.252 beta-NLL figure that a 130-run check found was not reproducible). No ratio is reported here for that reason: dividing by a noise-level baseline like +0.043 produces a multiplier that implies more precision than either number supports.

**Bottom line:** the spread-widening result is solid evidence that ensemble disagreement isn't collapsing the way self-reported confidence does. The correlation result is directionally encouraging — correctly signed, where notebook 2.0's single draw was not — but a multi-run check analogous to notebook 3.0's would be needed before treating -0.202 as an established improvement in magnitude over the self-reported approach's ≈ -0.09 mean.